<a href="https://colab.research.google.com/github/gbhav1003/python-1/blob/main/Database_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import sqlite3
import pandas as pd

 TASK 1: Database Design and Table Creation


In [15]:
print("=== TASK 1: Database Creation ===")

# Create connection and cursor
conn = sqlite3.connect('bookstore.db')
cursor = conn.cursor()
print("✓ Database connection established")

=== TASK 1: Database Creation ===
✓ Database connection established


In [16]:
cursor.execute("PRAGMA foreign_keys = ON;")

In [17]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER DEFAULT 0
)''')
print("✓ Books table created successfully")

✓ Books table created successfully


In [18]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    city TEXT,
    join_date TEXT
)''')
print("✓ Customers table created successfully")

✓ Customers table created successfully


In [19]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    book_id INTEGER,
    quantity INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
    FOREIGN KEY (book_id) REFERENCES Books(book_id)
)''')
print("✓ Orders table created successfully")

✓ Orders table created successfully


In [20]:
for table in ['Books', 'Customers', 'Orders']:
    print(f"\nSchema for {table} table:")
    schema = cursor.execute(f"PRAGMA table_info({table})").fetchall()
    for column in schema:
        print(column)


Schema for Books table:
(0, 'book_id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'author', 'TEXT', 1, None, 0)
(3, 'price', 'REAL', 1, None, 0)
(4, 'stock_quantity', 'INTEGER', 0, '0', 0)

Schema for Customers table:
(0, 'customer_id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'email', 'TEXT', 1, None, 0)
(3, 'city', 'TEXT', 0, None, 0)
(4, 'join_date', 'TEXT', 0, None, 0)

Schema for Orders table:
(0, 'order_id', 'INTEGER', 0, None, 1)
(1, 'customer_id', 'INTEGER', 0, None, 0)
(2, 'book_id', 'INTEGER', 0, None, 0)
(3, 'quantity', 'INTEGER', 1, None, 0)
(4, 'order_date', 'TEXT', 1, None, 0)
(5, 'total_amount', 'REAL', 0, None, 0)


TASK 2: Data Insertion and Querying

In [21]:
print("\n=== TASK 2: Data Insertion and Querying ===")

books_data = [
    ('Python Programming', 'John Smith', 599.99, 25),
    ('Data Science Handbook', 'Jane Doe', 899.50, 15),
    ('Machine Learning Basics', 'Alan Turing', 1299.00, 10),
    ('SQL Essentials', 'Edgar Codd', 499.99, 30),
    ('Web Development', 'Tim Berners', 799.00, 20)

]

customers_data = [
    ('Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15'),
    ('Priya Patel', 'priya@email.com', 'Delhi', '2024-01-20'),
    ('Amit Kumar', 'amit@email.com', 'Bangalore', '2024-02-01'),
    ('Sneha Reddy', 'sneha@email.com', 'Hyderabad', '2024-02-10'),
    ('Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')
]

orders_data = [
    (1, 1, 2, '2024-03-01', 1199.00),
    (1, 2, 1, '2024-03-02', 899.50),
    (2, 1, 1, '2024-03-03', 599.99),
    (2, 3, 1, '2024-03-05', 1299.00),
    (3, 4, 3, '2024-03-07', 1499.97),
    (4, 2, 1, '2024-03-10', 899.50),
    (5, 5, 2, '2024-03-12', 1598.00)
]


=== TASK 2: Data Insertion and Querying ===


In [22]:
cursor.executemany("INSERT OR IGNORE INTO Books (title, author, price, stock_quantity) VALUES (?, ?, ?, ?)", books_data)
cursor.executemany("INSERT OR IGNORE INTO Customers (name, email, city, join_date) VALUES (?, ?, ?, ?)", customers_data)
cursor.executemany("INSERT OR IGNORE INTO Orders (customer_id, book_id, quantity, order_date, total_amount) VALUES (?, ?, ?, ?, ?)", orders_data)
conn.commit()

In [23]:
print(f"✓ Inserted {len(books_data)} books, {len(customers_data)} customers, and {len(orders_data)} orders.")

✓ Inserted 5 books, 5 customers, and 7 orders.


In [24]:
conn = sqlite3.connect('bookstore.db')

df_books = pd.read_sql("SELECT * FROM Books", conn)
print(df_books.to_string(index=False))


 book_id                   title      author   price  stock_quantity
       1      Python Programming  John Smith  599.99              25
       2   Data Science Handbook    Jane Doe  899.50              15
       3 Machine Learning Basics Alan Turing 1299.00              10
       4          SQL Essentials  Edgar Codd  499.99              30
       5         Web Development Tim Berners  799.00              20


In [25]:
print("\nCustomers from Mumbai:")
for row in cursor.execute("SELECT * FROM Customers WHERE city = 'Mumbai'"):
    print(row)


Customers from Mumbai:
(1, 'Rahul Sharma', 'rahul@email.com', 'Mumbai', '2024-01-15')
(5, 'Vikram Singh', 'vikram@email.com', 'Mumbai', '2024-02-15')


In [26]:
print("\nBooks priced > 800 with stock > 10:")
for row in cursor.execute("SELECT * FROM Books WHERE price > 800 AND stock_quantity > 10"):
    print(row)


Books priced > 800 with stock > 10:
(2, 'Data Science Handbook', 'Jane Doe', 899.5, 15)


In [27]:
total_orders = cursor.execute("SELECT COUNT(*) FROM Orders").fetchone()[0]
print(f"\nTotal Orders: {total_orders}")


Total Orders: 7


In [28]:
top_customer = cursor.execute('''
    SELECT name, COUNT(order_id) as order_count
    FROM Customers JOIN Orders ON Customers.customer_id = Orders.customer_id
    GROUP BY name ORDER BY order_count DESC LIMIT 1
''').fetchone()
print(f"Top Customer: {top_customer[0]} ({top_customer[1]} orders)")

Top Customer: Rahul Sharma (2 orders)


In [29]:
total_revenue = cursor.execute("SELECT SUM(total_amount) FROM Orders").fetchone()[0]
print(f"Total Revenue: ₹{total_revenue:.2f}")

Total Revenue: ₹7994.96


# TASK 3: Pandas Integration and Analysis

In [30]:
print("\n=== TASK 3: Pandas Integration ===")

# Load to DataFrames
df_books = pd.read_sql("SELECT * FROM Books", conn)
df_customers = pd.read_sql("SELECT * FROM Customers", conn)
df_orders = pd.read_sql("SELECT * FROM Orders", conn)

report = df_orders.merge(df_customers, on='customer_id').merge(df_books, on='book_id')
report = report[['order_id', 'name', 'city', 'title', 'quantity', 'total_amount']]
report.columns = ['Order ID', 'Customer Name', 'City', 'Book Title', 'Qty', 'Total']

print("\nComprehensive Order Report (First 3):")
print(report.head(3))


=== TASK 3: Pandas Integration ===

Comprehensive Order Report (First 3):
   Order ID Customer Name    City             Book Title  Qty    Total
0         1  Rahul Sharma  Mumbai     Python Programming    2  1199.00
1         2  Rahul Sharma  Mumbai  Data Science Handbook    1   899.50
2         3   Priya Patel   Delhi     Python Programming    1   599.99


In [31]:
print(f"\nAverage Order Value: ₹{df_orders['total_amount'].mean():.2f}")
print("\nOrders by City:")
print(report['City'].value_counts())


Average Order Value: ₹1142.14

Orders by City:
City
Mumbai       3
Delhi        2
Bangalore    1
Hyderabad    1
Name: count, dtype: int64


In [32]:
discounts_data = pd.DataFrame({
    'book_id': [1, 2, 3, 4, 5],
    'discount_percent': [10, 15, 5, 20, 12]
})
discounts_data.to_sql('discounts', conn, if_exists='replace', index=False)
print("\n✓ Discounts table created via Pandas")


✓ Discounts table created via Pandas


In [33]:
query = '''
SELECT b.title, b.price as original_price, d.discount_percent,
       (b.price * (1 - d.discount_percent/100.0)) as discounted_price
FROM Books b
JOIN discounts d ON b.book_id = d.book_id
'''
print("\nBooks with Discounted Prices:")
print(pd.read_sql(query, conn))

# Close connection
conn.close()


Books with Discounted Prices:
                     title  original_price  discount_percent  discounted_price
0       Python Programming          599.99                10           539.991
1    Data Science Handbook          899.50                15           764.575
2  Machine Learning Basics         1299.00                 5          1234.050
3           SQL Essentials          499.99                20           399.992
4          Web Development          799.00                12           703.120
